In [1]:
import numpy as np
import serial
import time
import matplotlib.pyplot as plt
import concurrent.futures
import serial.tools.list_ports
import pandas as pd
import os
import re
from copy import deepcopy

ports = serial.tools.list_ports.comports()
ambit_ports = [x[0] for x in ports if "CH9102" in x[1]]
print("Available ports:", ambit_ports)
import logging
logger = logging.getLogger("serial")
logger.setLevel(logging.INFO)

Available ports: ['COM7']


In [2]:
def readline_from_port(ser: serial.Serial, init_wait: float = 0.25, line_wait: float = 0.5) -> tuple[str, bool]:
    '''
    Read a single line from the serial port. Waits for data to be available before reading. Timeout if no data is received within the specified wait times.
    Parameters:
        ser: serial.Serial
            The serial port to read from.
        init_wait: float
            The initial wait time for data to be available.
        line_wait: float
            The wait time for the rest of the line to be available.
    Returns:
        tuple[str, bool]
            str: The line read from the serial port.
            bool: True if more data is available to read, False otherwise.
    '''
    # initial wait
    _t = time.perf_counter()
    while ser.in_waiting == 0:
        if time.perf_counter() - _t > init_wait + .05:
            return "", False
        time.sleep(0.025)

    _str:str = ""
    _t = time.perf_counter()


    while True:
        if ser.in_waiting > 0:
            
            c = ser.read()
            if (c == bytes([0])) or (c > bytes([127])):
                continue           
            
            _t = time.perf_counter()
            c = c.decode('ascii')
            _str += c
            if c == '\n':
                return _str, ser.in_waiting > 0

        elif time.perf_counter() - _t > line_wait + .05:
            return _str, False
        else:
            time.sleep(0.04)

    return _str, False


def readlines_from_port(ser: serial.Serial, kw_stop = True, **argv) -> tuple[list[str], int]:
    

    line_to_line_wait = argv.get("line_to_line_wait", 2.0)
    chuck_time = argv.get("chuck_time", 999)
    chuck_size = argv.get("chuck_size", 10)
    timeout = argv.get("timeout", -1)
    ret_int = 0

    lines: list[str] = []
    _t0_start = time.perf_counter()
    kw = ["NEW Name Here Ready", "Done"]
    if timeout > 0: 
        _time_quick = _t0_start + timeout
    else:
        _time_quick = -1

    _t_start = time.perf_counter()
    _counter = 0
    while True:
        ret, more = readline_from_port(ser, init_wait=argv.get("init_wait", 0.25))
        if ret != "":
            lines.append(ret)
            _t_start = time.perf_counter()
            _counter += 1

        if kw_stop:
            if ret.strip() in kw:
                logger.info(f"Found stop keyword: {ret.strip()}")
                ret_int = -1
                break

        # premature stops
        if ((time.perf_counter() - _t0_start) > chuck_time) or (_counter >= chuck_size):
            ret_int = 2
            # logger.info(f"prematured chunk{time.perf_counter() - _t0_start} {_counter}")
            break

        if more:
            ret_int = 1
            continue

        # timeout
        if (time.perf_counter() - _t_start) > line_to_line_wait:
            # logger.info(f"line to line timeout{(time.perf_counter() - _t_start)}")
            ret_int = 0
            break

        if _time_quick > 0 and (time.perf_counter() > _time_quick):
            # logger.info(f"total timeout{(time.perf_counter() - _t0_start)}")
            ret_int = 0
            break
        
    return lines, ret_int


parse_data_line = re.compile(r'T:([\-\.0-9]+),F:([\.0-9]+),S:([0-9]+),R:([0-9]+),Sun:([0-9]+),L:([0-9]+)\n')
def parse_data(lines:list[str])->np.ndarray:
    _data = []
    for line in lines:
        match = parse_data_line.match(line)
        if match:
            _data.append(np.fromiter(match.groups(), dtype=float))
    return np.array(_data)

In [3]:
with serial.Serial(ambit_ports[0], 115200, timeout=1) as ser1:
    ser1.write(("hello"+'\n\r').encode())
    print(readlines_from_port(ser1, True))

(['NEW Name Here Ready\r\n'], -1)


In [5]:
%matplotlib qtagg

In [6]:
class Trace():
    cmd_str: str    
    run_seq: list[list[int]]
    _cmd_arr: np.ndarray
    mea_tml: np.ndarray
    mea_act: np.ndarray
    label: str



    timeline: np.ndarray
    data: np.ndarray
    time_start: float
    time_end: float
    time_t0:float

        
    def __init__(self, label:str = "", led_persist:bool = False):
        self.run_seq = []
        self.data = np.array([])
        self._led_persist = led_persist
        self.label = label

    def add_line(self, num:int, freq:int, actinic:int = 0):
        self.run_seq.append([num, freq, actinic])
        self.compile(self._led_persist)
        self.calc_timeline()

    def compile(self, persist:bool = False):
        run_seq = []
        for line in self.run_seq:
            num, freq, actinic = line
            run_seq.append([2, 0, num//256, num%256, freq//256, freq%256, actinic, 1])

        self._cmd_arr = np.reshape(run_seq, (-1, 8))
        arr_length = self._cmd_arr.shape[0]
        string = np.array2string(self._cmd_arr.flatten(), separator=',')[1:-1].replace(" ", "")
        self.cmd_str = "arrun1," + str(arr_length) + "," + str(int(persist)) + ","+ string + ",\n"

    def calc_timeline(self):
        _t, mea_tml, mea_act = 0, [], []
        for _n, _f, _a in self.run_seq:
            mea_tml.append(np.arange(_n) / _f + _t)
            mea_act.append([_a] * _n)
            _t = mea_tml[-1][-1]
        self.mea_tml = np.concatenate(mea_tml).astype(np.float64) * 0.854
        self.mea_act = np.concatenate(mea_act).astype(np.int8)

    def copy(self):
        t_new = Trace()
        t_new.run_seq = self.run_seq.copy()
        t_new._cmd_arr = self._cmd_arr.copy()
        t_new.cmd_str = self.cmd_str
        t_new.mea_tml = self.mea_tml.copy()
        t_new.mea_act = self.mea_act.copy()
        return t_new   
        
        


t1 = Trace()
t1.add_line(50, 2, 0)
# t1.add_line(50, 20, 50)
t1.add_line(50, 2, 0)

t1.compile()
t1.calc_timeline()
# print(t1.mea_tml)
# print(t1.mea_act)

In [7]:
result_lists = []

In [8]:
SAT_INTENSITY = 240
ACT_INTENSITY = 10
sat_Flash_0 = Trace("sat", False)
sat_Flash_0.add_line(20, 10, 0) # baseline
sat_Flash_0.add_line(100, 100, SAT_INTENSITY) # 1s Sat
sat_Flash_0.add_line(20, 100, 0) # decay1
sat_Flash_0.add_line(20, 10, 0) # decay2



Steady_state_0 = Trace("SS0", False)
Steady_state_0.add_line(10, 5, 0)

Steady_state_1 = Trace("SS1", False)
Steady_state_1.add_line(100, 5, 0)

qE_induction_0 = Trace("qE induction", True)
qE_induction_0.add_line(40, 20, ACT_INTENSITY) # 2s
qE_induction_0.add_line(100, 100, SAT_INTENSITY)  # 1s
qE_induction_0.add_line(40, 20, ACT_INTENSITY) # 2s


qE_relaxation_0 = Trace("qE relaxation", False)
qE_relaxation_0.add_line(40, 10, 0) # 4s
qE_relaxation_0.add_line(100, 100, SAT_INTENSITY)  # 2s
qE_relaxation_0.add_line(40, 5, 0) # 8s




In [214]:
runlist = [Steady_state_0.copy()] + [qE_induction_0.copy() for _ in range(3)] + [qE_relaxation_0.copy() for _ in range(3)]

In [9]:
runlist = [sat_Flash_0, Steady_state_1.copy(), Steady_state_1.copy()]

In [10]:
fig, ax = plt.subplots(1, 1, figsize=(8,4))
t0 = time.time()
y_range = [np.inf, -np.inf]
with serial.Serial(ambit_ports[0], 115200) as ser1:
    for trace in runlist:
        ser1.write((trace.cmd_str+'\n\r').encode())
        trace.time_start = time.time()
        current_trace = ax.plot([], [])[0]
        x1 = trace.mea_tml + trace.time_start - t0
        ax.set_xlim(-1, x1[-1])
        _data = []
        while True:
            lines, status = readlines_from_port(ser1, True, chuck_time = 1, line_to_line_wait=1)
            if len(lines) > 0:
                _line = parse_data(lines)
                if len(_line) > 0:
                    _data.append(_line)
            y1 = np.vstack(_data)[:, 1]
            current_trace.set_data(x1[:y1.shape[0]], y1)
            if min(y1) < y_range[0]: y_range[0] = min(y1)
            if max(y1) > y_range[1]: y_range[1] = max(y1)
            ax.set_ylim(*y_range)
            fig.canvas.draw()
            fig.canvas.flush_events() #
            if status < 0: break
        data = np.vstack(_data)
        trace.data = data
result_lists.append(deepcopy(runlist))

In [192]:
for trace in runlist:
    plt.plot(trace.mea_tml + trace.time_start, trace.data[:,1])

In [199]:

for res in result_lists:
    x, y = [], []
    for trace in res:
        x.append(trace.mea_tml + trace.time_start)
        y.append(trace.data[:,1])
    x = np.concatenate(x)
    y = np.concatenate(y)
    plt.plot(x - x[0], y)